# Lesson 1.3 — Tracing One Tomato

We run Perceive and Understand for real, emit a per-stage **trace** (`stage | reads -> writes | owner`), and verify the contract chain.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, WorldState,
    model_perception, understand, LAYER_REGISTRY)
checks = []
world = GreenhouseWorld.demo_row(n=6, seed=3)
st = WorldState(q=world.q.copy())
trace = []


In [2]:
# STAGE 1 — Perceive
st.detections = model_perception(world, rng=np.random.default_rng(3), noise=0.01)
st.tool_xy = world.tool_xy()
trace.append(('Perceive', 'world -> detections(%d)' % len(st.detections), st.owner_of('detections')))
# STAGE 2 — Understand
st.targets, st.current_target = understand(st.detections, world)
tgt = st.current_target['id'] if st.current_target else None
trace.append(('Understand', 'detections -> current_target=%s' % tgt, st.owner_of('current_target')))
for s, io, owner in trace:
    print(f'{s:11s} | {io:34s} | {owner}')


Perceive    | world -> detections(6)             | Perceive (Module 3 perception)
Understand  | detections -> current_target=F1    | Understand / Recover


### Check the contract chain
A correct trace: Understand's output is one of Perceive's detections, and it is reachable (Understand's postcondition).

In [3]:
det_ids = [d['id'] for d in st.detections]
checks.append(len(st.detections) >= 1)                       # Perceive produced something
checks.append(st.current_target is None or st.current_target['id'] in det_ids)
checks.append(st.current_target is None or st.current_target['reachable'])
# localise a None target the way the lesson teaches
if st.current_target is None:
    cause = 'empty detections (Perceive)' if not st.detections else 'no ripe+reachable (Understand)'
    print('current_target is None -> cause:', cause)
else:
    print('picked target:', st.current_target['id'], 'reachable=', st.current_target['reachable'])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


picked target: F1 reachable= True
3/3 checks passed.
All checks passed.
